# Week 7 — Principal Component Analysis (PCA)
## Global Life Expectancy Paradox

**Aim:** reduce the 15 original numeric predictors into a smaller set of principal components while preserving as much predictor variance as possible.

**PCA overview:** PCA forms new, mutually orthogonal variables (principal components) as linear combinations of the original variables. Components are ordered by the amount of variance they explain. Since variables use different units, standardize predictors before PCA.

**Important:** PCA is unsupervised; it does not use the life-expectancy target to choose directions. We exclude the target and the male/female life-expectancy fields from PCA inputs to avoid target leakage.

In [ ]:
from google.colab import files
import io
uploaded = files.upload()  # Upload the supplied feature-engineered CSV.
filename = next(iter(uploaded))
import pandas as pd, numpy as np
df_all = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Uploaded shape:", df_all.shape)
display(df_all.head())

## Step 1 — Prepare and standardize predictor features

In [ ]:
# Use only the original cleaned columns. Do not use precomputed *_scaled or *_minmax
# columns: doing so would duplicate information and distort PCA.
base_cols = [
    "name","region","iso2","population","pop_density","life_expectancy_male",
    "life_expectancy_female","sex_ratio","gdp_per_capita","gdp_growth",
    "unemployment","infant_mortality","fertility","homicide_rate",
    "urban_population","internet_users","secondary_school_enrollment_female",
    "secondary_school_enrollment_male","co2_emissions","forested_area"
]
df = df_all[base_cols].copy()
df["avg_life_expectancy"] = (df["life_expectancy_male"] + df["life_expectancy_female"]) / 2
target = "avg_life_expectancy"
predictors = [
    "population","pop_density","sex_ratio","gdp_per_capita","gdp_growth","unemployment",
    "infant_mortality","fertility","homicide_rate","urban_population","internet_users",
    "secondary_school_enrollment_female","secondary_school_enrollment_male",
    "co2_emissions","forested_area"
]
X = df[predictors].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
y = df[target].astype(float)
print(f"Rows: {len(df)} | candidate predictor features: {len(predictors)}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Standardized matrix shape:", X_scaled.shape)

## Step 2 — Fit PCA and inspect variance
Fit all components first so we can determine how many are needed to cross 90% and 95% cumulative explained variance.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
pca = PCA()
X_pca_all = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)
variance_df = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "cumulative_explained_variance": cumulative
})
display(variance_df)
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
n95 = int(np.searchsorted(cumulative, 0.95) + 1)
print("Components for >=90% variance:", n90)
print("Components for >=95% variance:", n95)

## Step 3 — Visualize individual and cumulative variance

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4.5))
plt.bar(range(1, len(explained)+1), explained)
plt.xlabel("Principal component"); plt.ylabel("Explained variance ratio")
plt.title("Variance Explained by Each Principal Component")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(cumulative)+1), cumulative, marker="o")
plt.axhline(.90, linestyle="--", label="90% threshold")
plt.axhline(.95, linestyle=":", label="95% threshold")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.ylim(0, 1.03); plt.grid(True, alpha=.25); plt.legend()
plt.title("Cumulative Explained Variance")
plt.tight_layout(); plt.show()

## Step 4 — Create reduced PCA representation
Use the smallest number of components that reaches 90% variance. Export component scores and the variance table. Component names are mathematical axes, not directly interpretable health indicators.

In [ ]:
n_components = n90
pca_reduced = PCA(n_components=n_components)
scores = pca_reduced.fit_transform(X_scaled)
pc_df = pd.DataFrame(scores, columns=[f"PC{i+1}" for i in range(n_components)])
pc_df.insert(0, "name", df["name"].values)
pc_df.insert(1, "avg_life_expectancy", y.values)
pc_df.to_csv("pca_reduced_features.csv", index=False)
variance_df.to_csv("pca_explained_variance.csv", index=False)
from google.colab import files
files.download("pca_reduced_features.csv")
files.download("pca_explained_variance.csv")